In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path().absolute().parent))

import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats

con = duckdb.connect("../data/processed/nba.duckdb")

df = con.execute("SELECT * FROM model_features").df()
print(f"Shape: {df.shape}")
print(f"Seasons: {df['SEASON'].min()} to {df['SEASON'].max()} ({df['SEASON'].nunique()} seasons)")
print(f"\nColumns ({len(df.columns)}):\n{df.columns.tolist()}")

focus_features = [
    'rs_net_rating',
    'rs_off_rating',
    'rs_def_rating',
    'rs_vs_top_teams_win_pct',
    'playoff_readiness_score',
]

print("\nFocus feature availability:")
print(df[focus_features].notna().sum().rename('non_null_rows').to_frame())

display(df.head(10))



In [ ]:
# Does physicality score predict playoff rounds reached?
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Physicality score vs rounds reached
axes[0].scatter(df['physicality_score'], df['playoff_rounds_reached'], alpha=0.4)
axes[0].set_xlabel('Physicality Score')
axes[0].set_ylabel('Playoff Rounds Reached')
axes[0].set_title('Physicality vs Playoff Depth')

# Add trend line
import numpy as np
z = np.polyfit(df['physicality_score'].fillna(0), df['playoff_rounds_reached'].fillna(0), 1)
p = np.poly1d(z)
x_line = np.linspace(df['physicality_score'].min(), df['physicality_score'].max(), 100)
axes[0].plot(x_line, p(x_line), 'r--', alpha=0.8)

# Plot 2: Distribution of physicality score by rounds reached
df_clean = df.dropna(subset=['physicality_score', 'playoff_rounds_reached'])
for rounds in sorted(df_clean['playoff_rounds_reached'].unique()):
    subset = df_clean[df_clean['playoff_rounds_reached'] == rounds]['physicality_score']
    axes[1].hist(subset, alpha=0.5, label=f'Round {int(rounds)}', bins=15)
axes[1].set_xlabel('Physicality Score')
axes[1].set_ylabel('Count')
axes[1].set_title('Physicality Distribution by Round')
axes[1].legend()

# Plot 3: Mean physicality score by rounds reached
mean_by_round = df_clean.groupby('playoff_rounds_reached')['physicality_score'].mean()
axes[2].bar(mean_by_round.index.astype(int), mean_by_round.values, color='steelblue')
axes[2].set_xlabel('Playoff Rounds Reached')
axes[2].set_ylabel('Mean Physicality Score')
axes[2].set_title('Avg Physicality by Round Reached')
axes[2].axhline(y=0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# Correlation
corr = df_clean['physicality_score'].corr(df_clean['playoff_rounds_reached'])
print(f"Correlation: physicality_score vs playoff_rounds_reached = {corr:.3f}")

In [ ]:
# Correlation of ALL features vs playoff rounds reached
numeric_cols = df.select_dtypes(include='number').columns.tolist()
exclude = ['TEAM_ID', 'playoff_rounds_reached']
feature_cols = [c for c in numeric_cols if c not in exclude]

correlations = df[feature_cols].corrwith(df['playoff_rounds_reached']).sort_values()

plt.figure(figsize=(10, 12))
correlations.plot(kind='barh', color=['red' if x < 0 else 'steelblue' for x in correlations])
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.title('Feature Correlation with Playoff Rounds Reached')
plt.xlabel('Correlation')
plt.tight_layout()
plt.show()

print("\nTop 5 positive correlations:")
print(correlations.tail(5))
print("\nTop 5 negative correlations:")
print(correlations.head(5))

In [ ]:
# Show actual calculated values for every feature for a few familiar teams
teams_to_show = ['LAL', 'GSW', 'SAS', 'MIA', 'BOS']

sample = df[df['TEAM_ABBR'].isin(teams_to_show)].sort_values(['TEAM_ABBR', 'SEASON'])

# Show one season per team for readability
latest = sample.groupby('TEAM_ABBR').last().reset_index()

# All numeric features
numeric_cols = df.select_dtypes(include='number').columns.tolist()
exclude = ['TEAM_ID']
feature_cols = ['TEAM_ABBR', 'SEASON'] + [c for c in numeric_cols if c not in exclude]

display(latest[feature_cols].T)

In [ ]:
# Drop leakage columns and metadata
exclude = [
    'TEAM_ID', 'TEAM_ABBR', 'SEASON', 'playoff_games',
    'po_games_played', 'games_played', 'playoff_rounds_reached',
    'foul_rate_playoff', 'ft_rate_playoff', 'dreb_pct_playoff',
    'po_ppg', 'po_off_rating', 'po_def_rating', 'po_net_rating'
]

target = 'playoff_rounds_reached'
feature_cols = [
    c for c in df.select_dtypes(include='number').columns
    if c not in exclude + [target]
]

df_clean = df[feature_cols + [target]].dropna()

print(f"Features evaluated: {len(feature_cols)}")
print(f"Samples used: {len(df_clean)}")



In [ ]:
# Correlation + statistical significance for every feature
results = []
for col in feature_cols:
    corr, pval = stats.pearsonr(df_clean[col], df_clean[target])
    results.append({
        'feature': col,
        'correlation': corr,
        'abs_correlation': abs(corr),
        'p_value': pval,
        'significant': pval < 0.05
    })

results_df = pd.DataFrame(results).sort_values('abs_correlation', ascending=False)

print("Feature correlations with rounds_reached (sorted by strength):\n")
print(results_df.to_string(index=False))

In [ ]:
# Multicollinearity check — which features are too correlated with each other?
significant_features = results_df[results_df['significant']]['feature'].tolist()

corr_matrix = df_clean[significant_features].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    square=True,
    linewidths=0.5,
    vmin=-1, vmax=1
)
plt.title('Feature Correlation Matrix — Significant Features Only')
plt.tight_layout()
plt.show()

# Flag high correlation pairs
print("\nHighly correlated feature pairs (|r| > 0.7):")
for i in range(len(corr_matrix)):
    for j in range(i+1, len(corr_matrix)):
        val = corr_matrix.iloc[i, j]
        if abs(val) > 0.7:
            print(f"  {corr_matrix.index[i]} ↔ {corr_matrix.columns[j]}: {val:.3f}")

In [ ]:
final_features = [
    'rs_net_rating',
    'rs_off_rating',
    'rs_def_rating',
    'rs_vs_top_teams_win_pct',
    'rs_win_pct',
    'rs_close_game_win_pct',
    'rs_close_game_count',
    'dreb_pct',
    'rs_ppg',
]
missing = [f for f in final_features if f not in df.columns]
print('Missing from dataset:', missing if missing else 'None')

final_corr = df_clean[final_features].corr()

plt.figure(figsize=(11, 8))
sns.heatmap(
    final_corr,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    square=True,
    linewidths=0.5,
    vmin=-1,
    vmax=1,
)
plt.title('RS-Only Production Feature Set - Correlation Check')
plt.tight_layout()
plt.show()

tmp = final_corr.copy()
np.fill_diagonal(tmp.values, 0)
print('Max pairwise correlation (excluding diagonal):')
print(f"  {tmp.abs().max().max():.3f}")

rows = []
for f in final_features:
    corr, pval = stats.pearsonr(df_clean[f], df_clean[target])
    rows.append({
        'feature': f,
        'correlation': corr,
        'abs_correlation': abs(corr),
        'p_value': pval,
    })

feature_target_df = pd.DataFrame(rows).sort_values('abs_correlation', ascending=False)
print('\nRS-only feature correlations with target:')
display(feature_target_df)




In [ ]:
# Feature health summary for the RS-only production feature set
model_features = [
    'rs_net_rating',
    'rs_off_rating',
    'rs_def_rating',
    'rs_vs_top_teams_win_pct',
    'rs_win_pct',
    'rs_close_game_win_pct',
    'rs_close_game_count',
    'dreb_pct',
    'rs_ppg',
]
feature_health = pd.DataFrame({
    'non_null': df[model_features].notna().sum(),
    'null_pct': df[model_features].isna().mean().round(4),
    'mean': df[model_features].mean(numeric_only=True),
    'std': df[model_features].std(numeric_only=True),
    'min': df[model_features].min(numeric_only=True),
    'max': df[model_features].max(numeric_only=True),
}).sort_values('null_pct', ascending=False)

print('Feature health summary (RS-only model feature set):')
display(feature_health)

new_features = ['rs_net_rating', 'rs_off_rating', 'rs_def_rating', 'rs_vs_top_teams_win_pct']
trend = (
    df.groupby('SEASON')[new_features]
      .mean()
      .reset_index()
      .sort_values('SEASON')
)

print('Season-level averages for ratings and top-team performance:')
display(trend)

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
for ax, feat in zip(axes.ravel(), new_features):
    ax.plot(trend['SEASON'], trend[feat], marker='o')
    ax.set_title(f'{feat} (playoff-team season mean)')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Availability profile kept for context
avail = con.execute(
    """
    SELECT
        a.SEASON,
        t.abbreviation AS team,
        a.rotation_size,
        a.avg_availability_rate,
        a.star_availability_rate,
        a.lineup_quality_score,
        a.depth_score,
        a.injury_games_lost,
        s.rounds_reached
    FROM availability_features a
    JOIN team_series_summary s
        ON a.TEAM_ID = s.team_id AND a.SEASON = s.season
    LEFT JOIN teams t
        ON a.TEAM_ID = t.id
    ORDER BY a.SEASON DESC, s.rounds_reached DESC
    """
).df()

print(f"Availability shape: {avail.shape}")
print("\nChampions availability profile:")
print(avail[avail['rounds_reached'] == 5][[
    'SEASON', 'team', 'rotation_size',
    'avg_availability_rate', 'star_availability_rate',
    'injury_games_lost'
]].to_string())

con.close()


